In [1]:
from gitsource import GithubRepositoryDataReader

reader = GithubRepositoryDataReader(
    repo_owner="DataTalksClub",
    repo_name="llm-zoomcamp",
    commit_id="8c1834d",
    allowed_extensions={"md"},
    filename_filter=lambda path: "/lessons/" in path,
)
documents = [file.parse() for file in reader.read()]

In [2]:
docs_tot_number = len(documents)
print(docs_tot_number)

documents[0]

72


{'content': '# Introduction\n\nVideo: [Watch this lesson](https://www.youtube.com/watch?v=rQYyFxf1FWw&list=PL3MmuxUbc_hLZFNgSad56pDBKK8KO0XIv)\n\nIn this module, we\'ll build a working Retrieval-Augmented\nGeneration (RAG) system from scratch, step by step.\n\nWe write everything in plain Python. We build a small search index by\nhand and call the LLM ourselves. I want you to see every piece first.\nThat way you know what a framework does for you before you reach for\none.\n\nPlaces where you can find me:\n\n- [My substack](https://alexeyondata.substack.com/)\n- [LinkedIn](https://www.linkedin.com/in/agrigorev/)\n- [X](https://x.com/Al_Grigor)\n\n## LLMs\n\nAn LLM (Large Language Model) is a neural network trained on massive\namounts of text. Given a prompt, it generates a continuation - a\nplausible next piece of text.\n\nThink of your phone. When you type "how are" in WhatsApp, it suggests\n"you" as the next word. "How are you" is the most common continuation.\nYour phone uses a simp

In [3]:
doc = documents[0]
print(doc["filename"])
print(doc["content"])

01-agentic-rag/lessons/01-intro.md
# Introduction

Video: [Watch this lesson](https://www.youtube.com/watch?v=rQYyFxf1FWw&list=PL3MmuxUbc_hLZFNgSad56pDBKK8KO0XIv)

In this module, we'll build a working Retrieval-Augmented
Generation (RAG) system from scratch, step by step.

We write everything in plain Python. We build a small search index by
hand and call the LLM ourselves. I want you to see every piece first.
That way you know what a framework does for you before you reach for
one.

Places where you can find me:

- [My substack](https://alexeyondata.substack.com/)
- [LinkedIn](https://www.linkedin.com/in/agrigorev/)
- [X](https://x.com/Al_Grigor)

## LLMs

An LLM (Large Language Model) is a neural network trained on massive
amounts of text. Given a prompt, it generates a continuation - a
plausible next piece of text.

Think of your phone. When you type "how are" in WhatsApp, it suggests
"you" as the next word. "How are you" is the most common continuation.
Your phone uses a simple la

Generating ground truth
==

In [4]:
data_gen_instructions = """
You emulate a student who is taking our LLM course.
You are given one lesson page from the course.
Formulate 5 questions this student might ask that are answered by this page.

Rules:
- The page should contain the answer to each question.
- Make the questions complete and not too short.
- Use as few words as possible from the page; don't copy its phrasing.
- The questions should resemble how people actually ask things online:
  not too formal, not too short, not too long.
- Ask about the content of the lesson, not about its formatting or filename.
""".strip()

from pydantic import BaseModel

class Questions(BaseModel):
    questions: list[str]

In [5]:
import json

user_prompt = json.dumps(doc)

In [6]:
user_prompt

'{"content": "# Introduction\\n\\nVideo: [Watch this lesson](https://www.youtube.com/watch?v=rQYyFxf1FWw&list=PL3MmuxUbc_hLZFNgSad56pDBKK8KO0XIv)\\n\\nIn this module, we\'ll build a working Retrieval-Augmented\\nGeneration (RAG) system from scratch, step by step.\\n\\nWe write everything in plain Python. We build a small search index by\\nhand and call the LLM ourselves. I want you to see every piece first.\\nThat way you know what a framework does for you before you reach for\\none.\\n\\nPlaces where you can find me:\\n\\n- [My substack](https://alexeyondata.substack.com/)\\n- [LinkedIn](https://www.linkedin.com/in/agrigorev/)\\n- [X](https://x.com/Al_Grigor)\\n\\n## LLMs\\n\\nAn LLM (Large Language Model) is a neural network trained on massive\\namounts of text. Given a prompt, it generates a continuation - a\\nplausible next piece of text.\\n\\nThink of your phone. When you type \\"how are\\" in WhatsApp, it suggests\\n\\"you\\" as the next word. \\"How are you\\" is the most common

In [7]:
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv()
openai_client = OpenAI()

In [18]:
messages = [
    {"role": "developer", "content": data_gen_instructions},
    {"role": "user", "content": user_prompt}
]

response = openai_client.responses.parse(
    model="gpt-5.4-mini",
    input=messages,
    text_format=Questions
)

In [19]:
result = response.output_parsed

print(result)

questions=['What is the main idea behind RAG, and why does it help with LLM answers?', 'Why does the course treat the language model like a black box instead of digging into how it works?', 'What are the main limits of LLMs that RAG is meant to fix?', 'What will you build in this module besides the FAQ chatbot itself?', 'What changes in the second part of the module when the system becomes agentic?']


In [8]:
from evaluation_utils import llm_structured

result, usage = llm_structured(
    openai_client,
    data_gen_instructions,
    user_prompt,
    Questions
)

print(result.questions)

['What does RAG do to help an LLM answer questions more reliably?', 'Why are LLMs described as black boxes in this lesson?', 'What are the main limitations of LLMs mentioned here?', 'What is the FAQ agent in this module supposed to answer questions about?', 'What topics will the first part of the module cover before the agent becomes more autonomous?']


In [16]:
print(doc["filename"])

01-agentic-rag/lessons/01-intro.md


In [9]:
usage.input_tokens, usage.output_tokens

(1020, 87)

Q1. Generating questions
==

In [17]:
def select_by_name(documents, doc_name):
    for doc in documents:
        if doc.get("filename") == doc_name:
            return doc
    return None

In [19]:
target_filename = "01-agentic-rag/lessons/02-environment.md"
selected_doc = select_by_name(documents, target_filename)


user_prompt = json.dumps(selected_doc)

result, usage = llm_structured(
    openai_client,
    data_gen_instructions,
    user_prompt,
    Questions
)

print(doc["filename"])
print(usage.input_tokens)
print(usage.output_tokens)
print(result.questions)

01-agentic-rag/lessons/01-intro.md
1286
120
['What do I need installed before starting this module, and what level of Python knowledge is expected?', 'How do I set up a new project for this course with uv and which packages should I add?', 'What’s the recommended way to keep my API key safe so I don’t commit it by accident?', 'How do I launch Jupyter and check that the OpenAI client is working in a notebook?', 'If I want to use Groq or another OpenAI-style API instead of OpenAI, what changes do I need to make?']


In [20]:
target_filename = "01-agentic-rag/lessons/03-rag.md"
selected_doc = select_by_name(documents, target_filename)


user_prompt = json.dumps(selected_doc)

result, usage = llm_structured(
    openai_client,
    data_gen_instructions,
    user_prompt,
    Questions
)

print(doc["filename"])
print(usage.input_tokens)
print(usage.output_tokens)
print(result.questions)

01-agentic-rag/lessons/01-intro.md
1753
90
['Why doesn’t a plain LLM answer course-specific questions well on its own?', 'How does adding FAQ text into the prompt help the model give the right answer?', 'What does RAG mean, and what are the retrieval and generation parts of it?', 'What are the main pieces in a basic RAG setup?', 'Why does search quality matter so much in a RAG system?']


Results of Q1
--


**01-agentic-rag/lessons/01-intro.md**

usage.input_tokens: 1020

usage.output_tokens: 87

**01-agentic-rag/lessons/02-environment.md**

usage.input_tokens: 1286

usage.output_tokens: 120

**01-agentic-rag/lessons/03-rag.md**

usage.input_tokens: 1753

usage.output_tokens: 90

Searching the chunks
--

In [2]:
from gitsource import chunk_documents

chunks = chunk_documents(documents, size=2000, step=1000)

In [4]:
print(len(chunks))
chunks[0]

295


{'start': 0,
 'content': '# Introduction\n\nVideo: [Watch this lesson](https://www.youtube.com/watch?v=rQYyFxf1FWw&list=PL3MmuxUbc_hLZFNgSad56pDBKK8KO0XIv)\n\nIn this module, we\'ll build a working Retrieval-Augmented\nGeneration (RAG) system from scratch, step by step.\n\nWe write everything in plain Python. We build a small search index by\nhand and call the LLM ourselves. I want you to see every piece first.\nThat way you know what a framework does for you before you reach for\none.\n\nPlaces where you can find me:\n\n- [My substack](https://alexeyondata.substack.com/)\n- [LinkedIn](https://www.linkedin.com/in/agrigorev/)\n- [X](https://x.com/Al_Grigor)\n\n## LLMs\n\nAn LLM (Large Language Model) is a neural network trained on massive\namounts of text. Given a prompt, it generates a continuation - a\nplausible next piece of text.\n\nThink of your phone. When you type "how are" in WhatsApp, it suggests\n"you" as the next word. "How are you" is the most common continuation.\nYour phon

In [5]:
# create the text index
from minsearch import Index

index = Index(
    text_fields=["content"],
    keyword_fields=["filename"]
)
index.fit(chunks)

In [6]:
from onnx_embedder.embedder import Embedder
embedder = Embedder()

# Embed in batches:
from tqdm.auto import tqdm
import numpy as np

batch_size = 50
X = []
chunks_contents = [chunk["content"] for chunk in chunks]

for i in tqdm(range(0, len(chunks_contents), batch_size)):
    batch = chunks_contents[i:i + batch_size]
    batch_vectors = embedder.encode_batch(batch)
    X.extend(batch_vectors)

X = np.array(X)

2026-07-13 21:28:16.517677090 [W:onnxruntime:Default, device_discovery.cc:133 GetPciBusId] Skipping pci_bus_id for PCI path at "/sys/devices/LNXSYSTM:00/LNXSYBUS:00/PNP0A03:00/device:07/VMBUS:01/5620e0c7-8062-4dce-aeb7-520c7ef76171" because filename "5620e0c7-8062-4dce-aeb7-520c7ef76171" did not match expected pattern of [0-9a-f]+:[0-9a-f]+:[0-9a-f]+[.][0-9a-f]+


  0%|          | 0/6 [00:00<?, ?it/s]

In [ ]:
# Creating the Vector Search index
from minsearch import VectorSearch

vindex = VectorSearch(keyword_fields=["filename"])
vindex.fit(X, chunks)

In [8]:
def text_search(query, num_results=5):
    results = index.search(query, num_results=num_results)
    return results

def vector_search(query, num_results=5):
    query_vector = embedder.encode(query)
    results = vindex.search(query_vector, num_results=num_results)
    return results

def rrf(result_lists, k=60, num_results=5):
    scores = {}
    docs = {}

    for results in result_lists:
        for rank, doc in enumerate(results):
            key = (doc["filename"], doc["start"])
            scores[key] = scores.get(key, 0) + 1 / (k + rank)
            docs[key] = doc

    ranked = sorted(scores, key=scores.get, reverse=True)
    return [docs[key] for key in ranked[:num_results]]

def hybrid_search(query, k=60):
    text_results = text_search(query, num_results=10)
    vector_results = vector_search(query, num_results=10)
    return rrf([text_results, vector_results], k=k)

Q2. First result with text search
==

In [14]:
import pandas as pd

df_ground_truth = pd.read_csv("ground-truth.csv")
ground_truth = df_ground_truth.to_dict(orient="records")

In [16]:

q = ground_truth[0]["question"]
print(q)

What exactly is a retrieval-augmented generation system, and why does it help with answers that the model wouldn't know on its own?


In [19]:
text_search_result = text_search(q, num_results=5)
print(text_search_result[0]["filename"])

01-agentic-rag/lessons/03-rag.md


Q3. First result with vector search
--

In [20]:
vector_search_result = vector_search(q, num_results=5)
print(vector_search_result[0]["filename"])

01-agentic-rag/lessons/01-intro.md


Evaluation metrics
--

In [22]:
def compute_relevance(q, search_function):
    filename_id = q["filename"]
    results = search_function(query=q["question"])

    relevance = []
    for d in results:
        relevance.append(int(d["filename"] == filename_id))

    return relevance

def compute_relevance_total(ground_truth, search_function):
    relevance_total = []

    for q in tqdm(ground_truth):
        relevance = compute_relevance(q, search_function)
        relevance_total.append(relevance)

    return relevance_total

In [24]:
relevance_sample = compute_relevance_total(ground_truth[0:5], text_search)
relevance_sample

  0%|          | 0/5 [00:00<?, ?it/s]

[[0, 0, 0, 0, 1],
 [1, 0, 1, 0, 0],
 [1, 1, 0, 0, 1],
 [1, 0, 1, 0, 0],
 [0, 0, 0, 0, 0]]

In [26]:
def hit_rate(relevance):
    cnt = 0

    for line in relevance:
        if 1 in line:
            cnt = cnt + 1

    return cnt / len(relevance)

def mrr(relevance):
    total_score = 0.0

    for line in relevance:
        for rank in range(len(line)):
            if line[rank] == 1:
                total_score = total_score + 1 / (rank + 1)
                break

    return total_score / len(relevance)

def evaluate(ground_truth, search_function):
    relevance_total = compute_relevance_total(ground_truth, search_function)

    return {
        "hit_rate": hit_rate(relevance_total),
        "mrr": mrr(relevance_total),
    }

Q4. Evaluating text search
--

In [27]:
evaluate(
    ground_truth,
    text_search
)

  0%|          | 0/360 [00:00<?, ?it/s]

{'hit_rate': 0.7583333333333333, 'mrr': 0.5942592592592594}

Q5. Evaluating vector search
--

In [28]:
evaluate(
    ground_truth,
    vector_search
)

  0%|          | 0/360 [00:00<?, ?it/s]

{'hit_rate': 0.725, 'mrr': 0.5486111111111112}